In [1]:

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/Tweets.csv")
df = df[["airline_sentiment", "text"]]
df.head()

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


In [2]:
import nltk
import string
import re
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
ps = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http.?://[^\s]+[\s]?', '', text)
    text = nltk.word_tokenize(text)
    y = []
    for i in text:
        if i not in stopwords.words('english'):
            y.append(i)
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

df['text_cleaned'] = df['text'].apply(clean_text)

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X = tfidf_vectorizer.fit_transform(df['text_cleaned']).toarray()
Y = df['airline_sentiment'].values

# Train Models
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

# Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Naive Bayes Accuracy: 0.7213114754098361
Random Forest Accuracy: 0.7523907103825137


In [3]:
!pip install streamlit
!pip install pyngrok #For exposing Streamlit app via a URL
!pip install streamlit ngrok transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 19.7 MB/s eta 0:00:00


In [4]:
!pip install streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 7s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [6]:
!wget -q -O - ipv4.icanhazip.com

34.10.189.167


In [9]:
from pyngrok import ngrok
ngrok.set_auth_token("2w7brj4ZrWEh7LEI55gKIA85NxV_Du3nG6bFSgDdbDb1Hit8")

In [18]:
%%writefile app.py
import streamlit as st
import pandas as pd
import nltk
import string
import re
from nltk.stem.porter import PorterStemmer

# NLTK downloads
nltk.download('stopwords')
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

ps = PorterStemmer()

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http.?://[^\s]+[\s]?', '', text)
    text = nltk.word_tokenize(text)
    y = []
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(ps.stem(i))
    return " ".join(y)

# Load dataset (replace with your dataset path)
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP/Tweets.csv")

# Clean text column
df['text_cleaned'] = df['text'].apply(clean_text)

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X = tfidf_vectorizer.fit_transform(df['text_cleaned']).toarray()
Y = df['airline_sentiment'].values

# Train-test split
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

# Train models
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

# Streamlit UI
st.title("✈️ Airline Sentiment Analysis")
st.write("This app uses Naive Bayes and Random Forest Classifiers to predict the sentiment of airline tweets")

# User Input
user_input = st.text_area("Enter a tweet about an airline")

if st.button("Predict Sentiment"):
    if user_input.strip() != "":
        # Preprocess input
        cleaned_input = clean_text(user_input)
        input_vector = tfidf_vectorizer.transform([cleaned_input]).toarray()

        # Predictions
        nb_prediction = nb_model.predict(input_vector)
        rf_prediction = rf_model.predict(input_vector)

        # Emoji map
        emoji_map = {
            'positive': '😊 🎉',
            'neutral': '😐',
            'negative': '😞 ❌'
        }

        # Display results
        st.write(f"Naive Bayes Prediction: {nb_prediction[0].capitalize()} {emoji_map.get(nb_prediction[0], '')}")
        st.write(f"Random Forest Prediction: {rf_prediction[0].capitalize()} {emoji_map.get(rf_prediction[0], '')}")

        # Display accuracy
        st.write("🔍 Naive Bayes Accuracy: ", accuracy_score(y_test, nb_model.predict(X_test)))
        st.write("🌟 Random Forest Accuracy: ", accuracy_score(y_test, rf_model.predict(X_test)))
    else:
        st.write("⚠️ Please enter a tweet for prediction.")

# Image Gallery
st.write("### Image Gallery")
image_paths = [
    '/content/drive/MyDrive/Sentiment Analysis/Positive.jpeg',
    '/content/drive/MyDrive/Sentiment Analysis/Neutral.png',
    '/content/drive/MyDrive/Sentiment Analysis/Negative.jpg'   # fixed typo
]

current_image = st.slider("Select Image", 1, len(image_paths), 1)
st.image(image_paths[current_image - 1], use_container_width =True)

Overwriting app.py


In [19]:
!streamlit run app.py &>/dev/null&
public_url = ngrok.connect(8501)
print("Streamlit App is running on:", public_url)


Streamlit App is running on: NgrokTunnel: "https://f83e488d3225.ngrok-free.app" -> "http://localhost:8501"
